## BASE

In [ ]:
import torch
import torch.nn as nn
from torchvision import transforms, datasets
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import f1_score
import time
from tqdm.notebook import tqdm
from collections import defaultdict
import numpy as np

In [ ]:
# DATA_ROOT = "/home/alex/internship/datasets/aqua20/data/aqua20"
# DATA_ROOT = "/Users/alex/Developpement/Internship/datasets/aqua20/data/aqua20"
DATA_ROOT = "/lustre/fswork/projects/rech/rbw/ucw75ke/datasets/aqua20/data/aqua20"
NUM_CLASSES = 20
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RESOLUTION = 252

In [3]:
transform = transforms.Compose([
    transforms.Resize(RESOLUTION),
    transforms.CenterCrop(RESOLUTION),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

In [4]:
backbone = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitb14')
backbone.eval()
for p in backbone.parameters():
    p.requires_grad = False
backbone = backbone.to(DEVICE)

Using cache found in /Users/alex/.cache/torch/hub/facebookresearch_dinov2_main
/Users/alex/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/Users/alex/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/Users/alex/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


In [ ]:
def extract_features(loader, desc="Extracting features"):
    all_feats, all_labels = [], []
    with torch.no_grad():
        for x, y in tqdm(loader, desc=desc, leave=False):
            all_feats.append(backbone(x.to(DEVICE)).cpu())
            all_labels.append(y)
    return torch.cat(all_feats), torch.cat(all_labels)

def train_linear_probe(train_feats, train_labels, test_feats, test_labels,
                       epochs=50, lr=1e-3, eval_every=10, seed=None):
    if seed is not None:
        torch.manual_seed(seed)
        np.random.seed(seed)
    head = nn.Linear(768, NUM_CLASSES).to(DEVICE)
    optimizer = torch.optim.Adam(head.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    g = torch.Generator()
    if seed is not None:
        g.manual_seed(seed)

    train_feat_loader = DataLoader(TensorDataset(train_feats, train_labels), batch_size=256, shuffle=True, generator=g)
    test_feat_loader = DataLoader(TensorDataset(test_feats, test_labels), batch_size=256, shuffle=False)

    history: dict[int, dict[str, float]] = {}

    for epoch in tqdm(range(epochs), desc="Training"):
        head.train()
        total_loss, correct, total = 0.0, 0, 0

        for feats, y in train_feat_loader:
            feats, y = feats.to(DEVICE), y.to(DEVICE)
            logits = head(feats)
            loss = criterion(logits, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(y)
            correct += (logits.argmax(dim=1) == y).sum().item()
            total += len(y)

        if (epoch + 1) % eval_every == 0 or epoch == epochs - 1:
            head.eval()
            all_preds, all_labels_val = [], []
            with torch.no_grad():
                for feats, y in test_feat_loader:
                    all_preds.append(head(feats.to(DEVICE)).argmax(dim=1).cpu())
                    all_labels_val.append(y)
            all_preds = torch.cat(all_preds).numpy()
            all_labels_val = torch.cat(all_labels_val).numpy()

            history[epoch + 1] = {
                "loss": total_loss / total,
                "train_acc": correct / total,
                "f1_macro": f1_score(all_labels_val, all_preds, average="macro"),
                "f1_weighted": f1_score(all_labels_val, all_preds, average="weighted"),
            }

            m = history[epoch + 1]
            tqdm.write(
                f"Epoch {epoch+1:>3}/{epochs} | Loss: {m['loss']:.4f} | "
                f"Train Acc: {m['train_acc']*100:.1f}% | "
                f"F1 Macro: {m['f1_macro']*100:.1f}% | F1 Weighted: {m['f1_weighted']*100:.1f}%"
            )
    return head, history

In [9]:
test_ds    = datasets.ImageFolder(f"{DATA_ROOT}/test",  transform=transform)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)
test_feats,    test_labels    = extract_features(test_loader,    "Test")

Test:   0%|          | 0/26 [00:00<?, ?it/s]

## Baselines

### Full Data

In [10]:
full_train = datasets.ImageFolder(f"{DATA_ROOT}/train", transform=transform)
full_loader = DataLoader(full_train, batch_size=64, shuffle=True)

In [13]:
print("\nTraining on full data...")
start_time = time.time()
full_feats,    full_labels    = extract_features(full_loader,    "Full train")
head_full, history_full = train_linear_probe(full_feats, full_labels, test_feats, test_labels,
                                epochs=50, eval_every=10)
full_total_time = time.time() - start_time
print(f"Training time: {full_total_time:.6f} seconds")


Training on full data...


Full train:   0%|          | 0/103 [00:00<?, ?it/s]

Training:   0%|          | 0/50 [00:00<?, ?it/s]

Epoch  10/50 | Loss: 0.1458 | Train Acc: 95.0% | F1 Macro: 89.6% | F1 Weighted: 90.9%


IndexError: list index out of range

## Setup multiple


In [ ]:
DISTILLED_BASE_DIR = "../logged_files/distillation/aqua20/dinov2_vitb"

In [ ]:
class DistilledRun():
    def __init__(self, distilled_run: str, batch_size: int = 20):
        self.distilled_run: str = distilled_run
        self.syn_data: dict[str, torch.Tensor] = torch.load(
            f"{DISTILLED_BASE_DIR}/{distilled_run}/data.pth", map_location="cpu"
        )
        self.distill_loader: DataLoader = DataLoader(
            TensorDataset(self.syn_data["images"], self.syn_data["labels"]),
            batch_size=batch_size, shuffle=False,
        )
        self.total_time: float = 0.0
        self.head: nn.Linear | None = None
        self.history: dict | None = None

    def run_training(self, test_feats, test_labels, epochs=50, lr=1e-3, eval_every=10, seed=None):
        start_time = time.time()
        feats, labels = extract_features(self.distill_loader, desc=f"Extracting features for {self.distilled_run}")
        self.head, self.history = train_linear_probe(
            feats, labels, test_feats, test_labels,
            epochs=epochs, lr=lr, eval_every=eval_every, seed=seed,
        )
        self.total_time = time.time() - start_time

    def to_dict(self):
        return {"total_time": self.total_time, "history": self.history}

In [ ]:
import re
from pathlib import Path

distilled_run_regex = re.compile(
    r"^distill_aqua20_h100_\d+ipc(?:_(?:physics|seathru|scramble))?(?:_s\d+)?$"
)

distilled_runs: list[DistilledRun] = [
    DistilledRun(p.name)
    for p in sorted(Path(DISTILLED_BASE_DIR).iterdir())
    if p.is_dir() and distilled_run_regex.match(p.name)
]
print(f"Found {len(distilled_runs)} distilled runs")

In [ ]:
import numpy as np
import pandas as pd

def parse_run_name(name: str) -> tuple[int | None, str, int | None]:
    if name == "full_data":
        return None, "full", None
    m = re.match(
        r"distill_aqua20_h100_(\d+)ipc(?:_(physics|seathru|scramble))?(?:_s(\d+))?$", name
    )
    if m is None:
        raise ValueError(f"run_name non reconnu : {name!r}")
    return int(m.group(1)), (m.group(2) or "baseline"), (int(m.group(3)) if m.group(3) else None)

def final_metrics(history, last_k=3):
    epochs = sorted(history.keys())[-last_k:]
    return {
        "f1_macro":    float(np.mean([history[e]["f1_macro"]    for e in epochs])),
        "f1_weighted": float(np.mean([history[e]["f1_weighted"] for e in epochs])),
        "train_acc":   float(np.mean([history[e]["train_acc"]   for e in epochs])),
    }

In [ ]:
ref = distilled_runs[0]
scores = []
for s in range(5):
    ref.run_training(test_feats, test_labels, epochs=50, seed=s)
    scores.append(final_metrics(ref.history)["f1_macro"] * 100)
print(f"Bruit sonde (même data.pth) : {np.mean(scores):.2f} ± {np.std(scores, ddof=1):.2f}")

In [ ]:
PROBE_SEED = 0

rows = []
for run in distilled_runs:
    ipc, variant, dseed = parse_run_name(run.distilled_run)
    run.run_training(test_feats, test_labels, epochs=50, lr=1e-3, eval_every=10, seed=PROBE_SEED)
    m = final_metrics(run.history)
    rows.append({"IPC": ipc, "Variant": variant, "dseed": dseed,
                 "F1 macro (%)": m["f1_macro"] * 100,
                 "F1 weighted (%)": m["f1_weighted"] * 100})

m_full = final_metrics(history_full)   # référence, n=1
rows.append({"IPC": None, "Variant": "full", "dseed": None,
             "F1 macro (%)": m_full["f1_macro"] * 100,
             "F1 weighted (%)": m_full["f1_weighted"] * 100})

df = pd.DataFrame(rows)

summary = (
    df.groupby(["IPC", "Variant"], observed=True, dropna=False)
      .agg(macro_mean=("F1 macro (%)", "mean"), macro_std=("F1 macro (%)", "std"),
           weighted_mean=("F1 weighted (%)", "mean"), weighted_std=("F1 weighted (%)", "std"),
           n=("F1 macro (%)", "count"))
      .reset_index()
)

variant_order = ["baseline", "physics", "seathru", "scramble", "full"]
summary["Variant"] = pd.Categorical(summary["Variant"], categories=variant_order, ordered=True)
summary = summary.sort_values(["IPC", "Variant"], na_position="last").reset_index(drop=True)

fmt = lambda mean, std: f"{mean:.2f}" if pd.isna(std) else f"{mean:.2f} ± {std:.2f}"
summary["F1 macro"]    = summary.apply(lambda r: fmt(r.macro_mean, r.macro_std), axis=1)
summary["F1 weighted"] = summary.apply(lambda r: fmt(r.weighted_mean, r.weighted_std), axis=1)
print(summary[["IPC", "Variant", "F1 macro", "F1 weighted", "n"]].to_string(index=False))